In [12]:
import json
import gzip

def amazon_metadata_parser(file_path):
    with gzip.open(file_path, 'rt', encoding='utf-8') as f:
        for line in f:
            try:
                data = json.loads(line)
                
                # 1. 카테고리 정제: 상위 4개까지만 유효한 분류로 인정
                # 샘플에서 본 것처럼 뒷부분의 광고성 키워드를 잘라냅니다.
                raw_categories = data.get('category', [])
                clean_categories = [c.strip() for c in raw_categories[:4]] 
                
                # 2. 너무 짧거나 의미 없는 데이터 필터링
                if not data.get('asin') or len(clean_categories) < 2:
                    continue
                
                yield {
                    "asin": data.get('asin'),
                    "title": data.get('title'),
                    "brand": data.get('brand', 'Unknown'),
                    "price": data.get('price', 'N/A'),
                    "categories": clean_categories  # 정제된 카테고리만 반환
                }
            except Exception:
                continue

# 재테스트
filepath = "./data/All_Amazon_Meta.json"
for product in amazon_metadata_parser(filepath):
    # 이제 훨씬 깔끔하게 출력될 것입니다.
    print(f"ASIN: {product['asin']} | Clean Categories: {' > '.join(product['categories'])}")
    break

ASIN: 6305121869 | Clean Categories: Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses


In [11]:
filepath = "./data/All_Amazon_Meta.json"

for product in amazon_metadata_parser(filepath):
    print(f"ASIN: {product['asin']} | Categories: {' > '.join(product['categories'])}")
    break # 샘플로 한 개만 출력

ASIN: 6305121869 | Categories: Clothing, Shoes & Jewelry > Women > Clothing > Tops, Tees & Blouses


In [3]:
import json
import gzip

# def amazon_metadata_parser(file_path):
#     # 우리가 타겟으로 하는 키워드들
#     target_categories = {"Computers", "Laptops", "Tablets", "Monitors", "Computer Components"}
    
#     with gzip.open(file_path, 'rb') as f:
#         for line in f:
#             try:
#                 data = json.loads(line)
#                 raw_categories = data.get('category', [])
                
#                 # 1. 필터링: 타겟 카테고리가 하나라도 포함되어 있는지 확인
#                 # set의 교집합(&)을 이용하면 속도가 매우 빠릅니다.
#                 if not (set(raw_categories) & target_categories):
#                     continue
                
#                 # 2. 카테고리 정제 (앞서 했던 대로 상위 4개만)
#                 clean_categories = [c.strip() for c in raw_categories[:4]]
                
#                 # 3. 필수 정보 확인
#                 asin = data.get('asin')
#                 title = data.get('title')
#                 if not asin or not title:
#                     continue
                
#                 yield {
#                     "asin": asin,
#                     "title": title,
#                     "brand": data.get('brand', 'Unknown'),
#                     "price": data.get('price', 'N/A'),
#                     "categories": clean_categories
#                 }
#             except Exception:
#                 continue

def amazon_metadata_parser(file_path):
    import gzip as gz
    import json
    
    # 찾고 싶은 핵심 단어들을 소문자로 정의
    target_keywords = {"computer", "laptop", "tablet", "monitor", "component"}
    
    print(f"DEBUG: Starting to scan {file_path}...")
    with gz.open(file_path, 'rb') as f:
        for i, line in enumerate(f):
            try:
                data = json.loads(line.decode('utf-8', errors='ignore'))
                
                # 1. category 필드가 있는지, 그리고 리스트인지 확인
                categories = data.get('category', [])
                if not isinstance(categories, list):
                    categories = [str(categories)] # 리스트가 아니면 리스트로 변환
                
                # 2. 모든 카테고리 텍스트를 하나로 합쳐서 소문자로 변환
                full_category_text = " ".join(categories).lower()
                
                # 3. 키워드 중 하나라도 포함되어 있는지 확인 (부분 일치 검사)
                if any(kw in full_category_text for kw in target_keywords):
                    yield {
                        "asin": data.get('asin'),
                        "title": data.get('title'),
                        "brand": data.get('brand', 'Unknown'),
                        "category": categories
                    }
            except Exception:
                continue

            # 너무 오래 걸릴 수 있으니 진행 상황 로그 추가 (중요!)
            if i % 50000 == 0 and i > 0:
                print(f"DEBUG: Still searching... Scanned {i} lines.")

# 테스트: 이제 노트북/컴퓨터 관련 데이터만 출력됩니다.
filepath = "./data/All_Amazon_Meta.json.gz"
for product in amazon_metadata_parser(filepath):
    print(f"✅ 필터링 성공: {product['title'][:50]}... | {product['categories']}")
    break

DEBUG: Starting to scan ./data/All_Amazon_Meta.json.gz...


In [8]:
import gzip
import shutil
import os

# 현재 경로 설정
INPUT_PATH = '/Users/chahyeon-yeong/Desktop/project/amazon-project/data/All_Amazon_Meta.json.gz'
OUTPUT_PATH = '/Users/chahyeon-yeong/Desktop/project/amazon-project/data/real_data.json'

def decompress_properly():
    print(f"--- 압축 해제 시작 (이 작업은 시간이 좀 걸릴 수 있습니다) ---")
    try:
        with gzip.open(INPUT_PATH, 'rb') as f_in:
            with open(OUTPUT_PATH, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        print(f"--- 해제 완료! 파일 생성됨: {OUTPUT_PATH} ---")
        print(f"파일 크기: {os.path.getsize(OUTPUT_PATH) / (1024**3):.2f} GB")
    except Exception as e:
        print(f"에러 발생: {e}")

if __name__ == "__main__":
    decompress_properly()

--- 압축 해제 시작 (이 작업은 시간이 좀 걸릴 수 있습니다) ---
--- 해제 완료! 파일 생성됨: /Users/chahyeon-yeong/Desktop/project/amazon-project/data/real_data.json ---
파일 크기: 11.48 GB


In [3]:
import gzip

FILE_PATH = '/Users/chahyeon-yeong/Desktop/project/amazon-project/data/real_data.json.gz'

def fast_check():
    print(f"--- 초고속 탐색 시작 ---")
    with gzip.open(FILE_PATH, 'rb') as f:
        for i in range(20):  # 딱 앞부분 20줄만 봅니다.
            line = f.readline()
            # 바이트 상태에서 바로 확인 (decode 안 함)
            if line.startswith(b'{'):
                print(f"🎯 찾았다! Line {i}번부터 진짜 데이터입니다.")
                print(f"내용 샘플: {line[:50]}...")
                return
            else:
                print(f"❌ Line {i}번은 데이터가 아닙니다: {line[:30]}")

if __name__ == "__main__":
    fast_check()

--- 초고속 탐색 시작 ---
🎯 찾았다! Line 0번부터 진짜 데이터입니다.
내용 샘플: b'{"category": ["Clothing, Shoes & Jewelry", "Women"'...


In [ ]:
import gzip
import json

# 에어플로우 경로가 아닌, 맥 로컬에 있는 실제 파일 경로를 넣으세요!
FILE_PATH = './data/All_Amazon_Meta.json.gz'

def test_read():
    print(f"--- 데이터 샘플 확인 시작 ---")

    with gzip.open(FILE_PATH, 'rb') as f_outer:
        with gzip.GzipFile(fileobj=f_outer, mode='rb') as f_inner:
            for i, line in enumerate(f_inner):
                if i >= 5: break  # 딱 5줄만 보자 
                try:
                    # 1. 원본 바이트 출력
                    #print(f"\n[Line {i} Raw Bytes]: {line[:100]}...") 
                    
                    # 2. JSON 파싱 후 키(Keys)들만 출력
                    data = json.loads(line.decode('utf-8', errors='ignore'))
                    print(f"[Line {i} Keys]: {list(data.keys())}")
                    
                    # 3. 전체 내용 출력 (데이터 구조 파악용)
                    print(f"[Line {i} Content]: {json.dumps(data, indent=2, ensure_ascii=False)[:300]}...")
                except Exception as e:
                    print(f"에러 발생: {e}")
    # with gzip.open(FILE_PATH, 'rb') as f:
    #     for i, line in enumerate(f):
    #         if i >= 5: break  # 딱 5줄만 보자
            
    #         try:
    #             # 1. 원본 바이트 출력
    #             print(f"\n[Line {i} Raw Bytes]: {line[:100]}...") 
                
    #             # 2. JSON 파싱 후 키(Keys)들만 출력
    #             data = json.loads(line.decode('utf-8', errors='ignore'))
    #             print(f"[Line {i} Keys]: {list(data.keys())}")
                
    #             # 3. 전체 내용 출력 (데이터 구조 파악용)
    #             print(f"[Line {i} Content]: {json.dumps(data, indent=2, ensure_ascii=False)[:300]}...")
    #         except Exception as e:
    #             print(f"에러 발생: {e}")
# def test_read():
#     target_keywords = ["computer", "laptop", "pc", "tablet", "electronics"]
#     count = 0
    
#     print(f"--- 로컬 파일 읽기 시작: {FILE_PATH} ---")
    
#     try:
#         with gzip.open(FILE_PATH, 'rb') as f:
#             for i, line in enumerate(f):
#                 # 10만 줄까지만 테스트
#                 if i > 100000: 
#                     print("\n10만 줄까지 검색했으나 더 이상 발견되지 않음.")
#                     break
                
#                 try:
#                     clean_line = line.strip()
#                     if not clean_line: continue
                    
#                     data = json.loads(clean_line.decode('utf-8', errors='ignore'))
#                     content_str = str(data).lower()
                    
#                     if any(kw in content_str for kw in target_keywords):
#                         count += 1
#                         print(f"찾았다! [{count}] ASIN: {data.get('asin')} | Title: {data.get('title')[:30]}...")
                        
#                         if count >= 10: # 10개만 찾으면 종료
#                             print("\n테스트 성공! 10개를 찾았습니다.")
#                             return
#                 except:
#                     continue
                    
#                 if i % 20000 == 0:
#                     print(f"{i}줄 분석 중...")
                    
#     except FileNotFoundError:
#         print("에러: 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")

if __name__ == "__main__":
    test_read()

--- 데이터 샘플 확인 시작 ---

[Line 0 Raw Bytes]: b'{"category": ["Clothing, Shoes & Jewelry", "Women", "Clothing", "Tops, Tees & Blouses", "Blouses & B'...
[Line 0 Keys]: ['category', 'tech1', 'description', 'fit', 'title', 'also_buy', 'image', 'tech2', 'brand', 'feature', 'rank', 'also_view', 'details', 'main_cat', 'similar_item', 'date', 'price', 'asin']
[Line 0 Content]: {
  "category": [
    "Clothing, Shoes & Jewelry",
    "Women",
    "Clothing",
    "Tops, Tees & Blouses",
    "Blouses & Button-Down Shirts",
    "Import",
    "Versatile Occasions - Great for Daily,Casual,I am sure you will like it!",
    "Black Friday Cyber Monday Christmas Loose Blouse V-Neck B...

[Line 1 Raw Bytes]: b'{"category": ["Clothing, Shoes & Jewelry", "Traditional & Cultural Wear", "Asian", "East Asian", "10'...
[Line 1 Keys]: ['category', 'tech1', 'description', 'fit', 'title', 'also_buy', 'image', 'tech2', 'brand', 'feature', 'rank', 'also_view', 'details', 'main_cat', 'similar_item', 'date', 'price', 'as

In [ ]:
import gzip, json
count = 0
with gzip.open('./data/All_Amazon_Meta.json.gz', 'rb') as f_outer:
    with gzip.open(f_outer, 'rt', encoding='utf-8') as f_inner:
        with open('electronics_sample.json', 'w') as f_out:
            for line in f_inner:
                data = json.loads(line)
                # 'category' 리스트 안에 Electronics가 있는지 확인
                if 'Electronics' in data.get('category', []):
                    f_out.write(json.dumps(data) + '\n')
                    count += 1
                #if count >= 1000000: break # 10만 개만 모이면 중단
print(f'추출 완료: {count}개')

추출 완료: 790039개


In [ ]:
import gzip, json

# 우리가 수집할 핵심 도메인 키워드
target_keywords = {'electronics', 'computers', 'cell phones', 'camera', 'audio'}

count = 0
with gzip.open('./data/All_Amazon_Meta.json.gz', 'rt', encoding='utf-8') as f_inner:
    with open('electronics_gold.json', 'w', encoding='utf-8') as f_out:
        for line in f_inner:
            data = json.loads(line)
            categories = data.get('category', [])
            
            # 리스트의 앞쪽 2개 항목 중 하나라도 키워드에 걸리면 추출
            # (계층 구조상 상위 카테고리에 집중)
            header_categories = [str(c).lower().strip() for c in categories[:2]]
            
            if any(kw in header_categories for kw in target_keywords):
                f_out.write(json.dumps(data) + '\n')
                count += 1
            
            if count >= 100000: break

print(f"✅ 추출 완료: {count}개")

In [33]:
import gzip, json
# count = 0
# with gzip.open('./data/meta_Electronics.json.gz', 'rt', encoding='utf-8') as f:
#     with open('electronics.json', 'w', encoding='utf-8') as f_out:
#             for line in f:
#                 data = json.loads(line)
#                 # 'category' 리스트 안에 Electronics가 있는지 확인
#                 categories = data.get('category',[])
#                 if categories and str(categories[0]).strip().lower() == 'electronics':
#                     f_out.write(json.dumps(data) + '\n')
#                     count += 1
#                 if count >= 300000: break # 10만 개만 모이면 중단
# print(f'추출 완료: {count}개')


# 설정
input_file = './data/meta_Electronics.json.gz'
output_file = 'electronics_ontology_sample.json'
max_samples = 30000  # 3만 개만 뽑기

count = 0
with gzip.open(input_file, 'rt', encoding='utf-8') as f_in:
    with open(output_file, 'w', encoding='utf-8') as f_out:
        for line in f_in:
            try:
                data = json.loads(line)
                categories = data.get('category', [])
                
                # 조건 1: 대분류가 Electronics일 것
                if categories and str(categories[0]).strip().lower() == 'electronics':
                    
                    # 조건 2: '브랜드' 정보가 있고 '연관 상품(related)' 정보가 있는 것 위주로!
                    # (이래야 온톨로지 관계선이 풍부해집니다)
                    if data.get('brand') and data.get('also_buy'):
                        f_out.write(json.dumps(data) + '\n')
                        count += 1
                
                if count >= max_samples:
                    break
            except:
                continue

print(f"✅ 정제 완료! {count}개의 고밀도 데이터가 {output_file}에 저장되었습니다.")

✅ 정제 완료! 30000개의 고밀도 데이터가 electronics_ontology_sample.json에 저장되었습니다.


In [1]:
import pandas as pd
df = pd.read_json('./data/electronics_ontology_sample.json', lines=True)

In [2]:
unique_category = set()

for r in df['category']:
    print(r.split(','))
    break

AttributeError: 'list' object has no attribute 'split'

In [3]:
import html

unique_category = set()

for r in df['category']:
    # r이 이미 리스트이므로 바로 순회합니다.
    if isinstance(r, list):
        for cat in r:
            # 1. 공백 제거 (strip)
            # 2. &amp; -> & 변환 (html.unescape)
            clean_cat = html.unescape(cat.strip())
            unique_category.add(clean_cat)

print(f"정제된 고유 카테고리 개수: {len(unique_category)}")
print(sorted(list(unique_category))[:10]) # 확인용

정제된 고유 카테고리 개수: 1125
['(2) Front Mounted Usb Ports', '1 Year Warranty', '1" high', '1,680D poly/900D poly', '10 hour talk time, 180 hour standby', '100% Genuine leather', '100% High Density Polyethylene', '11" high', '114 millimeters wide', '12" wide']


In [4]:
unique_category

{'Heatsinks',
 'Hi-Fi & HT Cabinets',
 'Personal Radios',
 'Camera Cine Dollies',
 '10 hour talk time, 180 hour standby',
 'TV Screen Protectors',
 'Subwoofer Amplifiers',
 'Marine Subwoofers',
 'Keyboard Cases',
 'Fabric/Nylon',
 'Rear protective flap holds shoulder straps in place, Padded handle and shoulder straps for ease of transport',
 'Televisions',
 'Works great with landlines and cell phones',
 'In-Dash Navigation',
 'Backgrounds',
 'synthetic',
 'Digital Pens',
 'LCD Displays',
 'Unique detachable chart wallet feature makes navigation a breeze',
 'Slide, Negative & Print Pages',
 'Vehicle Tracking and Monitoring Modules',
 'Slimline case protects your device on its own or in your favorite bag',
 'Roomy storage underneath flap for files, power brick and other bulk storage needs',
 'Video Surveillance',
 'Internal Blu-ray Drives',
 'Compatibility: Designed to fit a range of laptops, removable sleeve fits up to 15.4" in size; including models with an extended battery',
 'Pockets

In [73]:
unique_brand = set()
for cat in df['brand']:
            # 1. 공백 제거 (strip)
            # 2. &amp; -> & 변환 (html.unescape)
            clean_cat = html.unescape(cat.strip())
            unique_brand.add(clean_cat)

print("lenght", len(unique_brand))

lenght 4963


In [78]:
df['brand'].unique()

array(['33 Books Co.', "Visit Amazon's Carolina Garcia Aguilera Page",
       "Visit Amazon's Dick Gackenbach Page", ..., 'StretchScreenUSA',
       'AJC Battery', 'AJC'], dtype=object)

In [53]:
# 1. 'also_buy' 리스트를 튜플로 변환 (중복 체크를 위해)
df['also_buy_tuple'] = df['also_buy'].apply(lambda x: tuple(x) if isinstance(x, list) else ())

# 2. 'asin'과 'also_buy_tuple' 쌍으로 중복 제거
unique_df = df.drop_duplicates(subset=['asin', 'also_buy_tuple']).copy()

# 3. 다시 리스트로 복구 (필요한 경우) 및 불필요한 컬럼 삭제
unique_df['also_buy'] = unique_df['also_buy_tuple'].apply(list)
unique_df = unique_df[['asin', 'also_buy']]

print(f"원본 개수: {len(df)}, 중복 제거 후: {len(unique_df)}")

원본 개수: 30000, 중복 제거 후: 27005


In [55]:
unique_df.head()

,asin,also_buy
0,0043396828,[0999470906]
1,0060009810,"[0425167798, 039914157X]"
2,0060219602,"[0060219521, 0060219580, 0060219394]"
3,0070524076,"[0073049557, 0134454170, 1118142063, 007733968..."
4,0091912407,[0330509691]


In [56]:
import pandas as pd

# 1. 파일 로드 (전체 데이터를 다 읽습니다)
df = pd.read_json('./data/electronics_ontology_sample.json', lines=True)

# 2. 현재 우리가 DB에 넣은 전체 상품 ASIN 리스트 (비교 대상)
# (이미 27,005개로 필터링된 상태라면 그 df의 asin을 사용하세요)
existing_asins = set(df['asin'].unique())

# 3. also_buy 리스트 중 우리 파일 내에 존재하는 ASIN만 필터링하는 함수
def count_actual_matches(also_buy_list):
    if not isinstance(also_buy_list, list):
        return 0
    # 리스트 안의 상품 중 우리 파일(existing_asins)에 있는 것만 남김
    matches = [target for target in also_buy_list if target in existing_asins]
    return len(matches)

# 4. 계산 적용
df['actual_match_count'] = df['also_buy'].apply(count_actual_matches)

# 5. 결과 확인
total_file_links = df['actual_match_count'].sum()

print(f"📊 JSON 파일 내 전체 잠재적 also_buy 개수: {df['also_buy'].apply(lambda x: len(x) if isinstance(x, list) else 0).sum()}")
print(f"🔗 그 중 파일 내 상품끼리 연결된 실제 개수: {total_file_links}")

# 6. 실제 연결이 많은 상위 10개 상품 확인
print("\n[상위 10개 연결 상품]")
print(df[df['actual_match_count'] > 0][['asin', 'title', 'actual_match_count']].sort_values(by='actual_match_count', ascending=False).head(10))

📊 JSON 파일 내 전체 잠재적 also_buy 개수: 581326
🔗 그 중 파일 내 상품끼리 연결된 실제 개수: 59330

[상위 10개 연결 상품]
             asin                                              title  \
9024   B000LT0KIC              Plantronics Vista M22 Audio Processor   
6405   B0006FUHXO                       Plantronics AC Power Adapter   
10856  B0012CJ99S             Plantronics Ear Cushions Leatherette 2   
13480  B001S0UL3M  ZOOM 19785424 Training Adapter Switch for Head...   
10864  B0012EN2SK                   Plantronics 71782-01 Ear Cushion   
1778   B00009R76A                Dektol Paper Developer, 1Gallon mix   
6565   B00078NIUY  Plantronics 66735-01 Uniband CS50 Headband wit...   
5780   B00029RDGI               Plantronics A10 Direct Connect Cable   
2785   B00029RDGI               Plantronics A10 Direct Connect Cable   
17445  B00434OWTY  Brother 0.25&quot;x 26.25' Continuous Form Lab...   

       actual_match_count  
9024                   32  
6405                   31  
10856                  31  
13480  

In [58]:
# 1. DB에 들어간 것과 동일하게 유니크한 ASIN 리스트 생성 (27,005개)
unique_db_asins = set(df['asin'].unique()) 

# 2. 계산 함수 (타겟 상품이 저 27,005개 안에 있을 때만 카운트)
def count_strict_matches(also_buy_list):
    if not isinstance(also_buy_list, list):
        return 0
    # 파일 전체가 아니라, '유니크하게 필터링된 노드 목록'에 있는 것만 필터링
    matches = [target for target in also_buy_list]
    return len(matches)

# 3. 결과 확인
df_unique = df.drop_duplicates(subset=['asin']) # DB 상황과 동일하게 중복 제거
df_unique['strict_match_count'] = df_unique['also_buy'].apply(count_strict_matches)

total_sync_links = df_unique['strict_match_count'].sum()

print(f"✅ 중복 제거 후, 실제 DB 노드들끼리 맺을 수 있는 관계 수: {total_sync_links}")

✅ 중복 제거 후, 실제 DB 노드들끼리 맺을 수 있는 관계 수: 516497


/var/folders/s7/_6xph2kx7zj1t3c3tmb0gk5c0000gn/T/ipykernel_3120/2707883571.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_unique['strict_match_count'] = df_unique['also_buy'].apply(count_strict_matches)


In [8]:
def clean_categories(cat_list):
    if not cat_list:
        return "Unknown"
    
    # 1. 중첩 리스트일 경우 평탄화(Flatten)
    # 2. 중복 제거 (set)
    # 3. &amp; 같은 특수문자 정제
    flattened = []
    for sublist in cat_list:
        if isinstance(sublist, list):
            flattened.extend(sublist)
        else:
            flattened.append(sublist)
            
    # 중복 제거 및 문자열 합치기
    unique_cats = list(set(flattened))
    return ", ".join(unique_cats).replace("&amp;", "&")

product_data = {
    "title": "5.11 Radio Pouch",
    "categories": ['Electronics','eBook Readers &amp; Accessories','eBook Readers']
}

# 전처리 실행 -> "Electronics, Portable Audio, Accessories"
clean_cats = clean_categories(product_data['categories'])

In [9]:
clean_cats

'Electronics, eBook Readers & Accessories, eBook Readers'

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

# 1. 무료 모델 로드 (처음 실행 시 모델 다운로드로 시간이 조금 걸릴 수 있음)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. 테스트: "여성 취향"이라는 단어를 벡터로 변환
text = "여성용 로즈골드 스마트워치 세련된 디자인"
vector = embeddings.embed_query(text)

print(f"벡터 길이: {len(vector)}")  # 이 모델은 384차원을 사용합니다.
print(f"앞부분 일부: {vector[:5]}")

/Users/chahyeon-yeong/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


벡터 길이: 384
앞부분 일부: [0.02313404157757759, 0.053403981029987335, -0.02659614011645317, -0.02378927916288376, -0.039611052721738815]


In [ ]:
# 1. 사용자의 질문을 벡터로 변환 (무료 허깅페이스 모델 사용)
user_question = "여성 취향의 세련되고 고급스러운 시계 추천해줘"
query_vector = embeddings.embed_query(user_question)

# 2. Neo4j 벡터 검색 쿼리 실행
# top_k: 가장 유사한 상품 몇 개를 가져올지 결정 (여기선 3개)
search_query = """
CALL db.index.vector.queryNodes('product_embeddings', 3, $vector)
YIELD node AS product, score
RETURN 
    product.title AS title, 
    product.category AS category, 
    product.price AS price, 
    score
"""

results = graph.query(search_query, {"vector": query_vector})

# 3. 결과 출력
print(f"--- '{user_question}'에 대한 검색 결과 ---")
for res in results:
    print(f"[{round(res['score'] * 100, 1)}% 일치] {res['title']} ({res['price']}원)")